# Student D — D2 Delivery Partner Performance by Region

**Business Question:** Which delivery partners perform best in each region, and are lower-cost couriers actually cost-efficient after delays and cancellations?

This notebook generates the two final D2 charts:

1. **Exhibit D2.2 — WHO:** Grouped Bar Chart — Average Lead Days by Delivery Company × Region
2. **Exhibit D2.4 — WHEN:** Line Chart — Overall Monthly On-Time Delivery %

Exhibits **D2.1** and **D2.3** remain supporting SQL tables and do not require separate charts.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

YEAR = 2025

CSV_DIR = Path('task3_csv')
OUT_DIR = Path('charts')
OUT_DIR.mkdir(exist_ok=True)

required_files = [
    'd2_avg_lead_by_region_company.csv',
    'd2_monthly_ontime.csv'
]

print('Current folder :', Path.cwd())
print('CSV folder     :', CSV_DIR.resolve())
print('Charts folder  :', OUT_DIR.resolve())
print()

for file_name in required_files:
    file_path = CSV_DIR / file_name
    print(f'{file_name}:', 'FOUND' if file_path.exists() else 'MISSING')


## Figure 4.2.5 — Exhibit D2.2 / WHO: Average Lead Days by Delivery Company and Region

**Chart type:** Grouped Bar Chart

**Purpose:** Compares average delivery lead time across couriers within each destination region.

Lower lead days indicate faster delivery. This chart should be interpreted together with the D2.2 scorecard because speed alone does not account for cancellations or cost.


In [ ]:
lead = pd.read_csv(CSV_DIR / 'd2_avg_lead_by_region_company.csv')

lead['Avg Lead Days'] = pd.to_numeric(
    lead['Avg Lead Days'],
    errors='coerce'
)

pivot_lead = lead.pivot(
    index='Region',
    columns='Company',
    values='Avg Lead Days'
)

preferred_region_order = ['Central', 'East Coast', 'Northern', 'Southern']
region_order = [r for r in preferred_region_order if r in pivot_lead.index]
region_order += [r for r in pivot_lead.index if r not in region_order]
pivot_lead = pivot_lead.reindex(region_order)

ax = pivot_lead.plot(
    kind='bar',
    figsize=(13, 7),
    width=0.82
)

ax.set_title(f'Average Lead Days by Delivery Company and Region ({YEAR})')
ax.set_xlabel('Destination Region')
ax.set_ylabel('Average Lead Days')
ax.legend(
    title='Delivery Company',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    OUT_DIR / 'D2_Chart1_Avg_Lead_Days_By_Company_Region.png',
    dpi=220,
    bbox_inches='tight'
)
plt.show()


## Figure 4.2.6 — Exhibit D2.4 / WHEN: Overall Monthly On-Time Delivery Performance

**Chart type:** Line Chart

**Purpose:** Shows when overall delivery reliability improves or weakens across the year.

A single overall trend is used instead of six courier lines because **D2.2 already compares individual delivery companies**. This keeps D2.4 focused on the **WHEN** question.

Delivered volume should be read together with On-Time % so months with fewer observations are not over-interpreted.


In [ ]:
monthly = pd.read_csv(CSV_DIR / 'd2_monthly_ontime.csv')

monthly['Month Order'] = pd.to_numeric(
    monthly['Month Order'],
    errors='coerce'
)

monthly['Delivered'] = pd.to_numeric(
    monthly['Delivered'],
    errors='coerce'
)

monthly['On-Time'] = pd.to_numeric(
    monthly['On-Time'],
    errors='coerce'
)

monthly['On-Time %'] = pd.to_numeric(
    monthly['On-Time %'],
    errors='coerce'
)

monthly = monthly.sort_values('Month Order')

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    monthly['Month'],
    monthly['On-Time %'],
    marker='o'
)

ax.set_title(f'Overall Monthly On-Time Delivery Performance ({YEAR})')
ax.set_xlabel('Month')
ax.set_ylabel('On-Time Delivery (%)')
ax.set_ylim(0, 105)

for month, pct, delivered in zip(
    monthly['Month'],
    monthly['On-Time %'],
    monthly['Delivered']
):
    ax.annotate(
        f'{pct:.1f}%\n(n={int(delivered)})',
        (month, pct),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=8
    )

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig(
    OUT_DIR / 'D2_Chart2_Overall_Monthly_On_Time_Performance.png',
    dpi=220,
    bbox_inches='tight'
)
plt.show()


## Final D2 chart files

After **Run All**, the notebook saves:

- `D2_Chart1_Avg_Lead_Days_By_Company_Region.png`
- `D2_Chart2_Overall_Monthly_On_Time_Performance.png`

to the `charts` folder.

With the current Jupyter setup, this should normally resolve to:

`C:\Users\tpq11\charts`
